# Renewables Are Growing, but Fossil Fuels Still Dominate

## A Data Analysis of the Global Energy Transition (1990–2024)

This project analyzes the global energy transition using the OWID Energy Dataset.  
The analysis focuses on renewable energy growth, leading countries, fossil fuel dependence, and the relationship between renewable energy adoption and greenhouse gas emissions.

## Research Questions

1. How has the global share of renewable energy developed over time?
2. Which countries are leading the energy transition?
3. Which countries remain highly dependent on fossil fuels?
4. Is there a relationship between renewable electricity share and the carbon intensity of electricity generation?

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import country_converter as coco


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "owid_energy_data.csv"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)


## Dataset Overview

The OWID Energy Dataset contains country-level energy indicators covering renewable energy, fossil fuel consumption, greenhouse gas emissions, electricity generation and population statistics.

For this analysis, the focus is on renewable energy share, fossil fuel share and greenhouse gas emissions between 1990 and 2024.

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df[['renewables_share_energy',
    'renewables_share_elec',
    'fossil_share_energy',
    'greenhouse_gas_emissions']].describe()

In [ ]:
df['year'].value_counts().sort_index().tail(20)

In [ ]:
df[df['country'].str.contains('World', na=False)]

# Data Preparation

To answer the research questions, the dataset was filtered to include global observations between 1990 and 2024. Additional subsets were created for country-level and European analyses.

In [ ]:
world = df[df['country'] == 'World']

world

In [ ]:
world[['year',
       'renewables_share_energy',
       'renewables_share_elec',
       'fossil_share_energy']].tail(15)

In [ ]:
world = df[df['country'] == 'World'].copy()

world = world[
    (world['year'] >= 1990) &
    (world['year'] <= 2024)
]

In [ ]:
world[['year', 'renewables_share_energy']]

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    world['year'],
    world['renewables_share_energy'],
    label='Renewables'
)

plt.plot(
    world['year'],
    world['fossil_share_energy'],
    label='Fossil Fuels'
)

plt.title('Global Energy Mix: Renewables vs Fossil Fuels (1990–2024)')
plt.xlabel('Year')
plt.ylabel('Share of Total Energy (%)')

plt.legend()

plt.tight_layout()
plt.savefig(VISUALS_DIR / "1_global_energy_mix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    world['year'],
    world['renewables_share_energy'],
    linewidth=2
)

# mögliche Wendepunkte markieren
plt.axvline(x=2007, linestyle='--')
plt.axvline(x=2020, linestyle='--')

plt.title('Global Renewable Energy Share (1990–2024)')
plt.xlabel('Year')
plt.ylabel('Renewable Share of Total Energy (%)')

plt.tight_layout()
plt.savefig(VISUALS_DIR / "2_global_renewable_energy_share.png", dpi=150, bbox_inches="tight")
plt.show()


## Question 1: How Has the Global Share of Renewable Energy Developed?

### Key Findings

- The global share of renewable energy increased from 6.8% in 1990 to 14.8% in 2024.
- Growth accelerated noticeably after 2008.
- A small slowdown can be observed around 2020, followed by continued growth.
- Despite the steady increase in renewable energy, fossil fuels continue to dominate the global energy mix.

In [ ]:
df_2024 = df[df['year'] == 2024]

In [ ]:
df_2024 = df[df['year'] == 2024]

df_2024[
    ['country', 'iso_code', 'renewables_share_energy']
].sort_values(
    'renewables_share_energy',
    ascending=False
).head(20)

In [ ]:
countries_2024 = df_2024[
    df_2024['iso_code'].notna()
].copy()

countries_2024[
    ['country', 'renewables_share_energy']
].sort_values(
    'renewables_share_energy',
    ascending=False
).head(15)

In [ ]:
countries_2024[
    countries_2024['country'] == 'Germany'
][['country', 'renewables_share_energy']]

In [ ]:
countries_2024['renewables_share_energy'].median()

In [ ]:
countries_2024[
    countries_2024['country'] == 'Germany'
].index

In [ ]:
countries_2024[
    ['country', 'renewables_share_energy']
].sort_values(
    'renewables_share_energy',
    ascending=False
).head(10)

In [ ]:
top10 = countries_2024[
    ['country', 'renewables_share_energy']
].sort_values(
    'renewables_share_energy',
    ascending=False
).head(10)

germany = countries_2024[
    countries_2024['country'] == 'Germany'
][['country', 'renewables_share_energy']]

plot_df = pd.concat([top10, germany])

plot_df

In [ ]:
plot_df = plot_df.sort_values(
    'renewables_share_energy',
    ascending=True
)

plt.figure(figsize=(10,6))

colors = [
    'darkorange' if country == 'Germany' else 'seagreen'
    for country in plot_df['country']
]

plt.barh(
    plot_df['country'],
    plot_df['renewables_share_energy'],
    color=colors
)

plt.title('Top Renewable Energy Countries in 2024')
plt.xlabel('Renewable Energy Share (%)')
plt.ylabel('Country')

for index, value in enumerate(plot_df['renewables_share_energy']):
    plt.text(
        value - 2,
        index,
        f'{value:.1f}%',
        va='center',
        ha='right',
        color='white'
    )

plt.tight_layout()
plt.savefig(VISUALS_DIR / "3_top_renewable_energy_countries_2024.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
countries = df[df['iso_code'].notna()].copy()

countries_1990 = countries[
    countries['year'] == 1990
][['country', 'renewables_share_energy']]

countries_2024 = countries[
    countries['year'] == 2024
][['country', 'renewables_share_energy']]

In [ ]:
countries_1990.shape

In [ ]:
countries_2024.shape

In [ ]:
comparison = countries_1990.merge(
    countries_2024,
    on='country',
    suffixes=('_1990', '_2024')
)

comparison.shape

In [ ]:
comparison['improvement'] = (
    comparison['renewables_share_energy_2024']
    - comparison['renewables_share_energy_1990']
)

comparison.head()

In [ ]:
comparison[
    ['country', 'improvement']
].sort_values(
    'improvement',
    ascending=False
).head(15)

In [ ]:
comparison = comparison.dropna(
    subset=[
        'renewables_share_energy_1990',
        'renewables_share_energy_2024'
    ]
)

comparison.shape

In [ ]:
comparison.head()

In [ ]:
comparison[
    ['country', 'improvement']
].sort_values(
    'improvement',
    ascending=False
).head(15)

In [ ]:
top10_improvement = comparison[
    ['country', 'improvement']
].sort_values(
    'improvement',
    ascending=False
).head(10)

top10_improvement

In [ ]:
plot_df = top10_improvement.sort_values(
    'improvement',
    ascending=True
)

plt.figure(figsize=(10,6))

plt.barh(
    plot_df['country'],
    plot_df['improvement'],
    color='seagreen'
)

plt.title(
    'Countries with the Largest Increase in Renewable Energy Share (1990–2024)'
)

plt.xlabel('Increase in Renewable Energy Share (Percentage Points)')
plt.ylabel('Country')

for index, value in enumerate(plot_df['improvement']):
    plt.text(
        value - 1,
        index,
        f'{value:.1f}',
        va='center',
        ha='right',
        color='white'
    )

plt.tight_layout()
plt.savefig(VISUALS_DIR / "4_countries_largest_increase_renewable_energy.png", dpi=150, bbox_inches="tight")
plt.show()


## Question 2: Which Countries Are Leading the Energy Transition?

### Part 1: Countries with the Highest Share of Renewable Energy (2024)

#### Key Findings

* Iceland leads the ranking with a renewable energy share of 80.5%.
* Other leading countries include Norway (71.8%) and Sweden (51.3%).
* Many of the top-ranked countries benefit from favorable geographic conditions, such as hydropower or geothermal energy resources.
* Germany reaches 24.0% and ranks 21st worldwide.
* Despite its relatively lower ranking, Germany remains well above the global median of 13.0%.

### Part 2: Countries with the Largest Improvement Since 1990

#### Key Findings

* The largest increases in renewable energy share were achieved primarily by European countries.
* Denmark recorded the strongest improvement worldwide, increasing its renewable energy share by 40.5 percentage points.
* Germany ranks fifth globally, with an increase of 22.7 percentage points.
* Countries with the highest renewable energy shares today are not necessarily the countries that achieved the largest improvements since 1990.
* Differences in renewable energy progress can reflect geography, energy demand, infrastructure, policy, and other country-specific factors; this descriptive analysis does not isolate their individual effects.


In [ ]:
countries_2024 = df[
    (df['year'] == 2024)
    & (df['iso_code'].notna())
].copy()

In [ ]:
countries_2024[
    ['country', 'fossil_share_energy']
].sort_values(
    'fossil_share_energy',
    ascending=False
).head(20)

In [ ]:
countries_2024[
    countries_2024['country'] == 'Germany'
][['country', 'fossil_share_energy']]

In [ ]:
countries_2024['fossil_share_energy'].median()

In [ ]:
countries_2024[
    ['country', 'fossil_share_energy']
].sort_values(
    'fossil_share_energy',
    ascending=False
).reset_index(drop=True)

In [ ]:
ranking = countries_2024[
    ['country', 'fossil_share_energy']
].sort_values(
    'fossil_share_energy',
    ascending=False
).reset_index(drop=True)

ranking[
    ranking['country'] == 'Germany'
]

In [ ]:
top10_fossil = countries_2024[
    ['country', 'fossil_share_energy']
].sort_values(
    'fossil_share_energy',
    ascending=False
).head(10)

top10_fossil

In [ ]:
plot_df = top10_fossil.sort_values(
    'fossil_share_energy',
    ascending=True
)

plt.figure(figsize=(10,6))

plt.barh(
    plot_df['country'],
    plot_df['fossil_share_energy'],
    color='firebrick'
)

plt.title(
    'Most Fossil Fuel Dependent Countries (2024)'
)

plt.xlabel('Fossil Fuel Share (%)')
plt.ylabel('Country')

for index, value in enumerate(plot_df['fossil_share_energy']):
    plt.text(
        value - 1,
        index,
        f'{value:.1f}%',
        va='center',
        ha='right',
        color='white'
    )

plt.tight_layout()
plt.savefig(VISUALS_DIR / "5_most_fossil_fuel_2024.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Country-to-continent mapping for the European subset.


In [ ]:
countries_2024['continent'] = coco.convert(
    names=countries_2024['country'],
    to='continent'
)

In [ ]:
countries_2024[
    ['country', 'continent']
].sample(10)

In [ ]:
countries_2024['continent'].value_counts()

In [ ]:
europe_2024 = countries_2024[
    countries_2024['continent'] == 'Europe'
].copy()

europe_2024.shape

In [ ]:
europe_2024[
    ['country', 'fossil_share_energy']
].sort_values(
    'fossil_share_energy',
    ascending=False
).head(15)

In [ ]:
countries_2024[
    countries_2024['country'].isin(
        ['Netherlands', 'Germany']
    )
][[
    'country',
    'renewables_share_energy',
    'gas_share_energy',
    'oil_share_energy',
    'coal_share_energy',
    'fossil_share_energy'
]]

In [ ]:
top10_europe_fossil = europe_2024[
    ['country', 'fossil_share_energy']
].sort_values(
    'fossil_share_energy',
    ascending=False
).head(10)

top10_europe_fossil

In [ ]:
plot_df = top10_europe_fossil.sort_values(
    'fossil_share_energy',
    ascending=True
)

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    plot_df['country'],
    plot_df['fossil_share_energy'],
    color='firebrick'
)

plt.title(
    'Most Fossil Fuel Dependent Countries in Europe (2024)'
)

plt.xlabel('Fossil Fuel Share (%)')
plt.ylabel('Country')

for index, value in enumerate(plot_df['fossil_share_energy']):
    plt.text(
        value - 1,
        index,
        f'{value:.1f}%',
        va='center',
        ha='right',
        color='white'
    )

plt.tight_layout()
plt.savefig(VISUALS_DIR / "6_most_fossil_fuel_europe_2024.png", dpi=150, bbox_inches="tight")
plt.show()


## Question 3: Which Countries Remain Highly Dependent on Fossil Fuels?

### Global Perspective

#### Key Findings

* Several countries in the 2024 sample still obtain very large shares of their total energy from fossil fuels.
* Several countries still obtain almost all of their energy from fossil fuels in 2024.
* Turkmenistan, Trinidad and Tobago, and Kuwait reach fossil fuel shares close to 100%.

### European Perspective

#### Key Findings

* Russia has the highest fossil fuel dependence in Europe, with a fossil fuel share of 88.2%.
* Poland and Belarus follow closely, both exceeding 86%.
* Germany's fossil fuel share remains high at 76.0%. Although Germany is not among the ten most fossil-dependent countries in Europe, it continues to rely heavily on fossil energy sources.
* The results demonstrate that the energy transition is progressing at very different speeds across European countries.


In [ ]:
# Variables used for Question 4
[
    col for col in [
        "renewables_share_elec",
        "carbon_intensity_elec"
    ]
    if col in df.columns
]


In [ ]:
electricity_scatter = countries_2024[
    [
        "country",
        "renewables_share_elec",
        "carbon_intensity_elec"
    ]
].dropna().copy()

electricity_scatter.shape


In [ ]:
electricity_scatter[
    ["renewables_share_elec", "carbon_intensity_elec"]
].describe()


In [ ]:
global_corr = electricity_scatter[
    ["renewables_share_elec", "carbon_intensity_elec"]
].corr().iloc[0, 1]

global_corr


In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    electricity_scatter["renewables_share_elec"],
    electricity_scatter["carbon_intensity_elec"],
    alpha=0.7
)

plt.title(
    "Renewable Electricity Share vs Carbon Intensity (2024)"
)
plt.xlabel("Renewable Share of Electricity Generation (%)")
plt.ylabel("Carbon Intensity of Electricity (g CO₂e/kWh)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    VISUALS_DIR / "7_renewable_electricity_vs_carbon_intensity.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()


In [ ]:
electricity_scatter.sort_values(
    "carbon_intensity_elec",
    ascending=False
).head(15)


In [ ]:
europe_2024 = countries_2024[
    countries_2024["continent"] == "Europe"
].copy()


In [ ]:
europe_electricity_scatter = europe_2024[
    [
        "country",
        "renewables_share_elec",
        "carbon_intensity_elec"
    ]
].dropna().copy()

europe_electricity_scatter.shape


In [ ]:
europe_corr = europe_electricity_scatter[
    ["renewables_share_elec", "carbon_intensity_elec"]
].corr().iloc[0, 1]

europe_corr


In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    europe_electricity_scatter["renewables_share_elec"],
    europe_electricity_scatter["carbon_intensity_elec"],
    alpha=0.7
)

plt.title(
    "Europe: Renewable Electricity Share vs Carbon Intensity (2024)"
)
plt.xlabel("Renewable Share of Electricity Generation (%)")
plt.ylabel("Carbon Intensity of Electricity (g CO₂e/kWh)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    VISUALS_DIR / "8_europe_renewable_electricity_vs_carbon_intensity.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()


In [ ]:
europe_electricity_scatter.sort_values(
    "carbon_intensity_elec",
    ascending=False
).head(15)


In [ ]:
electricity_scatter[
    electricity_scatter["renewables_share_elec"] >= 50
].sort_values(
    "carbon_intensity_elec",
    ascending=True
).head(20)


In [ ]:
# Correlation coefficients are descriptive associations, not causal estimates.
print(f"Global Pearson correlation: {global_corr:.3f}")
print(f"Europe Pearson correlation: {europe_corr:.3f}")


## Question 4: Is There a Relationship Between Renewable Electricity Share and Carbon Intensity?

### Interpretation

* This section compares the share of electricity generated from renewable sources with the carbon intensity of electricity generation in 2024.
* Pearson correlation is used as a descriptive measure of association.
* A negative correlation means that countries with larger renewable electricity shares tend to have lower-carbon electricity systems in this cross-sectional sample.
* The relationship must not be interpreted as a causal estimate. Nuclear power, fossil-fuel composition, imports, hydro resources, demand patterns, and other structural factors can also affect electricity-sector carbon intensity.
* The analysis is repeated for Europe to check whether the pattern is similar within a more geographically comparable subset.


# Conclusion

The global energy transition has made measurable progress over the last three decades. The share of renewable energy in the global energy mix increased substantially between 1990 and 2024, while fossil fuels still account for the majority of total energy use.

Country-level comparisons show large differences in both current renewable shares and the scale of change since 1990. These rankings are descriptive and should be interpreted in the context of geography, energy demand, economic structure, and available technologies.

The final analysis examines the association between renewable electricity generation and the carbon intensity of electricity production. This comparison is more appropriate for the OWID Energy Dataset than treating its electricity-sector greenhouse-gas variable as total national emissions. The correlation is descriptive rather than causal and is supplemented with a separate European subset.

Overall, the project illustrates how global and country-level energy indicators can be combined to examine the pace and unevenness of the energy transition while keeping the interpretation within the limits of the underlying data.
